In [11]:
import pandas as pd

In [12]:
# loading the district 8 file and header argument


path = path = '../data/raw/station_5min/d08_text_station_5min_2026_02_05.txt.gz'


names = [
    'timestamp',
    'station',
    'district',
    'freeway',
    'direction',
    'lane_type',        # ML = mainline, OR = on-ramp, FR = off-ramp, HV = HOV
    'station_length',   # miles
    'samples',
    'pct_observed',     # 0 = all values imputed by PeMS
    'total_flow',       # vehicles per 5 min, all lanes
    'avg_occupancy',    # fraction of time a detector is occupied
    'avg_speed',        # mph
]


df = pd.read_csv(path, header=None, names=names, usecols=range(len(names)))

print(df.head())
print(df.dtypes)

             timestamp  station  district  freeway direction lane_type  \
0  02/05/2026 00:00:00   801230         8       10         W        ML   
1  02/05/2026 00:00:00   801231         8       10         W        OR   
2  02/05/2026 00:00:00   801232         8       10         E        ML   
3  02/05/2026 00:00:00   801233         8       10         E        OR   
4  02/05/2026 00:00:00   801234         8       10         W        ML   

   station_length  samples  pct_observed  total_flow  avg_occupancy  avg_speed  
0           0.181       39           100       199.0         0.0276       65.3  
1             NaN       20           100        14.0            NaN        NaN  
2           0.110       40           100       135.0         0.0296       69.7  
3             NaN       20           100        10.0            NaN        NaN  
4           0.177       30            75        75.0         0.0231       67.8  
timestamp             str
station             int64
district         

In [10]:
# converting timestamp to real datetime 

df['timestamp'] = pd.to_datetime(df['timestamp'], format='%m/%d/%Y %H:%M:%S')

print(df.dtypes['timestamp'])
print(df['timestamp'].min(), df['timestamp'].max())

datetime64[us]
2026-02-05 00:00:00 2026-02-05 23:55:00


In [15]:
# lane types 

print(df.groupby('lane_type')['station'].nunique())

lane_type
FR     213
HV     442
ML    1154
OR     366
Name: station, dtype: int64


In [16]:
# rows per station 

print(df.groupby('station').size().value_counts())

288    2175
Name: count, dtype: int64


In [20]:
# filter to the study network: mainline sensors on I-10, I-15, I-215, SR-60, SR-91
mainline = df[(df['lane_type'] == 'ML') & (df['freeway'].isin([10, 15, 215, 60, 91]))].copy()
print(mainline.shape[0])

# unique stations in the study network
mainline_stations = mainline['station'].unique()
print(len(mainline_stations))

# stations by freeway and direction
mainline.groupby(['freeway', 'direction'])['station'].nunique().unstack()



284256
987


direction,E,N,S,W
freeway,,,,
10,143.0,NaN,NaN,156.0
15,NaN,161.0,154.0,NaN
60,56.0,NaN,NaN,62.0
91,43.0,NaN,NaN,40.0
215,NaN,84.0,88.0,NaN


In [ ]:
# data quality: real mainline vs imputed values by pems 

station_obs = mainline.groupby('station')['pct_observed'].mean() 


# pems also filled out computed values so, 100 means real and 0 means fully imputed. We can bin the stations by the percentage of real values.
bins = pd.cut(station_obs, [-1, 0, 50, 90, 100],
              labels=['0% (fully imputed)', '1-50%', '51-90%', '91-100%'])
print(bins.value_counts().sort_index(), '\n')
print(f"Fully imputed: {(station_obs == 0).sum()} of {len(station_obs)} stations "
      f"({(station_obs == 0).mean():.0%})")


imputed = mainline[mainline['pct_observed'] == 0]
print(f"Imputed rows: {len(imputed):,} | missing speed: {imputed['avg_speed'].isna().sum()} "
      f"| mean speed: {imputed['avg_speed'].mean():.1f} mph")

pct_observed
0% (fully imputed)    463
1-50%                  42
51-90%                 61
91-100%               421
Name: count, dtype: int64 

Fully imputed: 463 of 987 stations (47%)
Imputed rows: 136,224 | missing speed: 0 | mean speed: 61.4 mph
